In [ ]:
import kagglehub
import pandas as pd
import mlflow
import mlflow.sklearn
from prefect import flow, task
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score
from imblearn.under_sampling import RandomUnderSampler
import os


@task(retries=2, retry_delay_seconds=60)
def download_data():
    """Descarga el dataset desde Kaggle usando kagglehub."""
    print("Descargando dataset...")
    path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
    print("Path..." + path)
    # El path suele ser una carpeta, buscamos el .csv

    for file in os.listdir(path):
        if file.endswith(".csv"):
            return os.path.join(path, file)

    print("file..." + file)


@task
def prepare_data(file_path):
    """Carga y prepara los datos para el entrenamiento."""
    df = pd.read_csv(file_path)
    X = df.drop(["Class", "Time"], axis=1)  # 'Time' suele no ser útil sin ingeniería
    y = df["Class"]

    # Manejo de desbalance: Undersampling para balancear rápido el experimento
    rus = RandomUnderSampler(random_state=42)
    X_res, y_res = rus.fit_resample(X, y)

    return train_test_split(X_res, y_res, test_size=0.2, random_state=42)


@task
def train_and_log_model(X_train, X_test, y_train, y_test):
    """Entrena el modelo y registra métricas/artefactos en MLflow."""
    with mlflow.start_run(run_name="RandomForest_UnderSample"):
        n_estimators = 100
        model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
        model.fit(X_train, y_train)

        # Predicciones
        y_pred = model.predict(X_test)
        auprc = average_precision_score(y_test, model.predict_proba(X_test)[:, 1])

        # Log de parámetros y métricas
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_metric("auprc", auprc)

        # Log del modelo
        mlflow.sklearn.log_model(model, "fraud-model")

        print(f"Modelo entrenado con AUPRC: {auprc:.4f}")
        return mlflow.active_run().info.run_id


@flow(name="Credit Card Fraud MLOps Pipeline")
def fraud_detection_pipeline():
    # 1. Ingesta
    csv_path = download_data()

    # 2. Preprocesamiento
    # X_train, X_test, y_train, y_test = prepare_data(csv_path)

    # 3. Entrenamiento y Tracking
    # run_id = train_and_log_model(X_train, X_test, y_train, y_test)

    # print(f"Pipeline completado. Run ID: {run_id}")


if __name__ == "__main__":
    fraud_detection_pipeline()

12:01:36.438 | INFO    | Flow run 'convivial-mackerel' - Beginning flow run 'convivial-mackerel' for flow 'Credit Card Fraud MLOps Pipeline'

Descargando dataset...
Path.../Users/diegofernandonunezdiaz/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3


12:01:36.980 | INFO    | Task run 'download_data-30c' - Finished in state Completed()

12:01:37.445 | INFO    | Flow run 'convivial-mackerel' - Finished in state Completed()